# E2B

<img src="./assets/aliyun-sandbox.svg">

函数计算：https://www.aliyun.com/product/fc

region对照表：https://help.aliyun.com/zh/ecs/user-guide/regions-and-zones

## 安装 E2B 依赖

In [ ]:
!uv add e2b

## 沙箱管理

### 创建沙箱

In [1]:
from langchain_python.core.config import sandbox_settings

sandbox_settings

SandboxSettings(api_key='e2b_bc19a3ffe9fa22e25d197f742d9d3b03e1b28782', api_url='https://api.cn-hangzhou.e2b.fc.aliyuncs.com', domain='cn-hangzhou.e2b.fc.aliyuncs.com', role_arn='acs:ram::1213573388370267:role/aliyunserviceroleforfc', oss_endpoint='oss-cn-beijing.aliyuncs.com', oss_bucket='pythonai', template='code-interpreter-v1')

In [8]:
from e2b import AsyncSandbox

# 准备好沙箱配置
basic_config = {
    "timeout": 300, # 从创建开始计时，时间到了自动销毁
    "api_key": sandbox_settings.api_key,
    "api_url": sandbox_settings.api_url,
    "domain": sandbox_settings.domain,
}

sandbox = await AsyncSandbox.create(
    template=sandbox_settings.template,
    **basic_config,
) 
sandbox.sandbox_id

'sbx-423641d5-ba94-48c3-8812-7274a49d58b0'

### 连接已有沙箱

In [9]:
sandbox = await AsyncSandbox.connect(
    sandbox_id=sandbox.sandbox_id, 
    **basic_config
)
sandbox.sandbox_id

'sbx-423641d5-ba94-48c3-8812-7274a49d58b0'

### 手动销毁沙箱

In [19]:
await sandbox.kill()

True

### 手动刷新时间

In [10]:
await sandbox.set_timeout(300) # 从现在起再重新计时300秒后销毁

## 沙箱操作

### 文件读写

In [11]:
sandbox = await AsyncSandbox.create(
    template=sandbox_settings.template,
    **basic_config
) 

In [12]:
file_path = "/home/user/workspace/test.txt"
await sandbox.set_timeout(300)
await sandbox.files.write(file_path, "Hello World!")

WriteInfo(name='test.txt', type=None, path='/home/user/workspace/test.txt', metadata=None)

In [13]:
await sandbox.set_timeout(300)
await sandbox.files.read(file_path)

'Hello World!'

### 命令执行

In [14]:
await sandbox.set_timeout(300)
await sandbox.commands.run("node --version")

CommandResult(stderr='', stdout='v20.20.2\n', exit_code=0, error=None)

In [15]:
await sandbox.set_timeout(300)
await sandbox.commands.run("npm init -y", cwd="/home/user/workspace")

CommandResult(stderr='', stdout='Wrote to /home/user/workspace/package.json:\n\n{\n  "name": "workspace",\n  "version": "1.0.0",\n  "main": "index.js",\n  "scripts": {\n    "test": "echo \\"Error: no test specified\\" && exit 1"\n  },\n  "keywords": [],\n  "author": "",\n  "license": "ISC",\n  "description": ""\n}\n\n\n\n', exit_code=0, error=None)

In [16]:
await sandbox.set_timeout(300)
await sandbox.commands.run("ls", cwd="/home/user/workspace", timeout=60)

CommandResult(stderr='', stdout='package.json\ntest.txt\n', exit_code=0, error=None)

## 挂载 OSS

In [ ]:
import json
metadata_config = {
    "metadata":{
        "fc.sandbox.storage.oss": json.dumps({
            "mountPoints": [
                {
                    "bucketName": sandbox_settings.oss_bucket,
                    "mountDir": "/home/user/workspace",
                    "bucketPath": "/e2b-test/workspace",
                    "endpoint": sandbox_settings.oss_endpoint,
                    "readOnly": False,
                },
                {
                    "bucketName": sandbox_settings.oss_bucket,
                    "mountDir": "/home/user/output",
                    "bucketPath": "/e2b-test/output",
                    "endpoint": sandbox_settings.oss_endpoint,
                    "readOnly": False,
                }
            ]
        }),
        "fc.sandbox.auth.role": sandbox_settings.role_arn,
    }
}

In [23]:
# 创建沙箱
sandbox = await AsyncSandbox.create(
    template=sandbox_settings.template,
    **basic_config, # type: ignore
    **metadata_config
)

SandboxException: 400: OSS mount requires executionRoleArn (set on template or via fc.sandbox.auth.role in metadata)

In [ ]:
# 创建文件
await sandbox.set_timeout(300)
await sandbox.files.write("/home/user/workspace/source.txt", "source")
await sandbox.files.write("/home/user/output/output.txt", "output")

In [ ]:
await sandbox.kill()

In [ ]:
# 新建沙箱
sandbox = await AsyncSandbox.create(
    template=sandbox_settings.template,
    **basic_config, # type: ignore
    **metadata_config
)

In [ ]:
await sandbox.set_timeout(300)
await sandbox.commands.run("ls", cwd="/home/user/workspace")

In [ ]:
await sandbox.set_timeout(300)
await sandbox.commands.run("ls", cwd="/home/user/output")

## 沙箱实战

In [ ]:
# 创建沙箱
sandbox = await AsyncSandbox.create(
    template=sandbox_settings.template,
    **basic_config, # type: ignore
    **metadata_config
)

In [ ]:
await sandbox.set_timeout(300)
await sandbox.commands.run(
    "npm create vite@latest my-vue-app -- --template vue --no-interactive", cwd="/home/user/workspace"
)
await sandbox.set_timeout(300)

In [ ]:
await sandbox.set_timeout(300)
await sandbox.commands.run(
    "npm i", cwd="/home/user/workspace/my-vue-app"
)
await sandbox.set_timeout(300)

In [ ]:
await sandbox.files.write("/home/user/workspace/my-vue-app/vite.config.js", """
import vue from '@vitejs/plugin-vue';
import { defineConfig } from 'vite';

// https://vite.dev/config/
export default defineConfig({
  plugins: [vue()],
  base: '',
});
""")

In [ ]:
await sandbox.set_timeout(300)

In [ ]:
await sandbox.set_timeout(300)
await sandbox.commands.run(
    "npm run build", cwd="/home/user/workspace/my-vue-app"
)
await sandbox.set_timeout(300)

In [ ]:
await sandbox.set_timeout(300)
await sandbox.commands.run(
    "cp -a /home/user/workspace/my-vue-app/dist /home/user/output/my-vue-app"
)
await sandbox.set_timeout(300)

In [ ]:
await sandbox.kill()